In [1]:
import sys
import os
import torch
import json
import random
import numpy as np
from google.colab import drive

# 1. Mount Google Drive direkte i eksperiment-notebooken
drive.mount("/content/drive")

# 2. Tilføj projektsti til Python
PROJECT_ROOT = "/content/drive/MyDrive/Søren og Magnus/Deep Learning /Project_deep"
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

# 3. Importer moduler fra src mappen
from src.dataset import build_dataloaders
from src.models import get_resnet18

from src.losses import get_loss_function
from src.trainer import train_one_epoch, evaluate

# 4. Konfiguration af eksperiment
CONFIG = {
    "experiment_id": "E0",
    "seeds": [0, 42, 123],
    "epochs": 10,
    "batch_size": 32,
    "lr": 1e-4,
    "img_size": 224,
    "num_classes": 15,
}

train_dir = "/content/data/train"
val_dir = "/content/data/val"
test_dir = "/content/data/test"
device = "cuda" if torch.cuda.is_available() else "cpu"

# 5. Sti til JSON-resultatfilen på Drive
output_path = os.path.join(PROJECT_ROOT, f"results_{CONFIG['experiment_id']}.json")

# 6. Tjek om der allerede findes et delvist/færdigt resultat på Drive
if os.path.exists(output_path):
    with open(output_path, "r") as f:
        saved_data = json.load(f)
        seed_results = saved_data.get("results", [])
    completed_seeds = [r["seed"] for r in seed_results]
    print(f"Fundet eksisterende resultater på Drive for seeds: {completed_seeds}")
else:
    seed_results = []
    completed_seeds = []

# 7. Multi-seed løkke med auto-resume
for seed in CONFIG["seeds"]:
    # Springer over hvis dette seed allerede blev gemt tidligere
    if seed in completed_seeds:
        print(f"\n--- Skipping {CONFIG['experiment_id']} | Seed: {seed} (Allerede gennemført) ---")
        continue

    print(f"\n--- Running {CONFIG['experiment_id']} | Seed: {seed} ---")

    # Sæt seeds for reproducerbarhed
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)

    # Bygger dataloaders (og udpakker zip hvis mappen er tom)
    train_loader, val_loader, test_loader, class_names = build_dataloaders(
        train_dir, val_dir, test_dir, CONFIG
    )

    model = get_resnet18(num_classes=CONFIG["num_classes"]).to(device)
    criterion = get_loss_function(CONFIG["experiment_id"], train_loader, device=device)
    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"])

    for epoch in range(CONFIG["epochs"]):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_acc, val_f1, _ = evaluate(model, val_loader, device)
        print(
            f"Epoch {epoch+1}/{CONFIG['epochs']} | Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}"
        )

    test_acc, test_f1, test_report = evaluate(model, test_loader, device)
    print(f"--> Test Acc: {test_acc:.4f} | Test Macro-F1: {test_f1:.4f}")

    # Tilføj resultatet for det aktuelle seed
    seed_results.append(
        {"seed": seed, "test_accuracy": test_acc, "test_macro_f1": test_f1, "class_report": test_report}
    )

    # GEM STRAKS TIL DRIVE EFTER HVERT SEED (Checkpointing)
    with open(output_path, "w") as f:
        json.dump({"config": CONFIG, "results": seed_results, "classes": class_names}, f, indent=4)
    print(f" Gemte resultater for seed {seed} direkte til Google Drive.")

print(f"\n Eksperiment {CONFIG['experiment_id']} er fuldstændig færdigt! Resultater ligger i: {output_path}")

IndentationError: unexpected indent (2063241475.py, line 25)